# G7 — What the hub does to local structure

The project has two halves that have not yet been made to talk to each
other.

**The science (G5):** what encoders share is *local* — neighbourhood-scale
agreement is large, global agreement is thin.

**The engineering (G1/G3):** the hub is a *global* linear projection —
whitened PCA plus one ridge map per encoder.

So how does the hub work at all? A linear map recovers global linear
structure, which G5 says is the thin part, yet the hub delivers 93–96%
component transfer. Two explanations have never been distinguished:

- **(a)** the map preserves each encoder's local neighbourhoods, and
  transfer works *because* local structure survives it;
- **(b)** the map discards local structure and transfer runs on the thin
  global component alone — in which case the ceiling the hub misses
  *is* the local structure.

These make opposite predictions, and three measurements separate them.

**Why this was not measured before.** The 21-pair analyses of G5 are
deliberately *map-free*: they show that shared structure exists prior to
any fitting. Measuring inside the hub confounds structure that was
already shared with structure the fit created — which is exactly why
that comparison had to come after, and separately.

**A dry run on the Experiment B vectors suggests the answer is neither
(a) nor (b) cleanly**, so read the numbers rather than expecting a
verdict.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np

SOURCES = {
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch.npz", "img"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch.npz",  "img"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch.npz", "img"),
    "txt_bge":   ("crossmodal_pairs.npz",                   "txt"),
    "txt_gpt2":  ("crossmodal_pairs_gpt2.npz",              "txt"),
    "txt_bert":  ("e13_txt_bert.npz",                       "txt"),
    "txt_sbert": ("e13_txt_sbert.npz",                      "txt"),
}
SPACES = {}
for name, (fn, key) in SOURCES.items():
    f = DATA_DIR / fn
    if f.exists():
        d = np.load(str(f))
        if key in d.files:
            SPACES[name] = d[key].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
names = list(SPACES)
NEV = 1000
tr = np.arange(N - NEV); te = np.arange(N - NEV, N)
NS = min(1500, len(te) + len(tr))
sub = np.random.default_rng(0).permutation(N)[:NS]

def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a=1e-2):
    return np.linalg.solve(X[tr].T @ X[tr] + a * np.eye(X.shape[1]),
                           X[tr].T @ Y[tr])
def knn(Z, k):
    S = Z @ Z.T; np.fill_diagonal(S, -9.0)
    return np.argpartition(-S, k, axis=1)[:, :k]
def overlap(X, Y, k=10):
    a, b = knn(X, k), knn(Y, k)
    return float(np.mean([len(set(a[i]) & set(b[i])) / k
                          for i in range(len(X))]))

def build_hub(d):
    """G1's recipe: standardise each block on TRAIN, concatenate,
    whitened PCA fitted on TRAIN rows only."""
    blocks, stats = [], []
    for k in names:
        M = SPACES[k]
        mu, sd = M[tr].mean(0), M[tr].std(0) + 1e-8
        blocks.append((M - mu) / sd); stats.append((mu, sd))
    Z = np.hstack(blocks)
    Zc = Z - Z[tr].mean(0)
    _, S, Vt = np.linalg.svd(Zc[tr], full_matrices=False)
    W = Vt[:d].T / (S[:d] / np.sqrt(len(tr)) + 1e-8)
    return Zc @ W

META_MOD = {k: ("image" if k.startswith("img") else "text") for k in names}
print(f"{len(names)} spaces, {N} rows, {len(tr)} train / {len(te)} eval")

## 1 · Does the map preserve each encoder's own neighbourhoods?

Before asking what the hub does *between* encoders, ask what it does to
one. Compare each encoder's native neighbour sets with its neighbour sets
after mapping into the hub. A gentle rotation would score near 1.0; a
projection that reshapes local structure will score well below.

In [ ]:
HUB_DIM = 512
H = build_hub(HUB_DIM)
TO_HUB = {k: ridge(SPACES[k], H) for k in names}
INHUB = {k: l2n(SPACES[k] @ TO_HUB[k]) for k in names}

print(f"neighbourhood preservation under the hub map (k=10, "
      f"HUB_DIM={HUB_DIM})\n")
print(f"{'encoder':12s} {'native -> in-hub':>18s}")
PRES = {}
for k in names:
    PRES[k] = overlap(l2n(SPACES[k][sub]), INHUB[k][sub])
    print(f"{k:12s} {PRES[k]:18.3f}")
print(f"\nmean {np.mean(list(PRES.values())):.3f}")
print("\nWell below 1.0 means the hub map is NOT a gentle rotation - it")
print("reshapes each encoder's local neighbourhood structure substantially")
print("while still supporting transfer. That is the first thing to know.")

## 2 · Does the hub create local agreement, or merely carry it?

Now the cross-encoder question. If the hub *aligns* local structure,
in-hub overlap should exceed native overlap. If it merely relocates
structure that was already shared, the two will be similar. If in-hub
overlap is *lower*, the hub is buying global alignment by spending local
structure — which would make the width cliff and the missing retrieval
ceiling the same phenomenon.

In [ ]:
print(f"cross-encoder neighbourhood overlap, native vs in-hub "
      f"(k=10)\n")
print(f"{'pair':26s} {'native':>8s} {'in-hub':>8s} {'change':>8s}")
DELTA = {}
for i, a in enumerate(names):
    for b in names[i+1:]:
        nat = overlap(l2n(SPACES[a][sub]), l2n(SPACES[b][sub]))
        hub = overlap(INHUB[a][sub], INHUB[b][sub])
        DELTA[(a, b)] = (nat, hub, hub - nat)
        print(f"{a+' - '+b:26s} {nat:8.3f} {hub:8.3f} {hub-nat:+8.3f}")
ds = [v[2] for v in DELTA.values()]

# ---- STRATIFY before averaging ----------------------------------------
# A mean over all pairs is contaminated: a COLLAPSED space (very low
# effective rank, near-degenerate neighbourhoods) gains enormously from
# the hub's whitening, because almost any reshaping improves on
# neighbourhoods that were arbitrary to begin with. That is the C.11
# isotropy rescue, not evidence that the hub aligns local structure in
# general. Report the two populations separately.
def eff_rank(M):
    C = np.cov(M[tr].T)
    ev = np.clip(np.linalg.eigvalsh(C), 1e-12, None)
    pr = ev / ev.sum()
    return float(np.exp(-(pr * np.log(pr)).sum()))

def pair_cos(M, n=1500, seed=0):
    """Mean cosine between RANDOM pairs. Near 0 = healthy, +0.999 =
    collapsed. Unlike effective rank / width this does not depend on
    the ambient dimension, so it does not confuse 'text encoder' with
    'degenerate' - text spaces legitimately have lower rank/width
    ratios than image spaces."""
    r = np.random.default_rng(seed)
    Z = M[r.permutation(len(M))[:n]]
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    S = Z @ Z.T
    iu = np.triu_indices(len(Z), 1)
    return float(S[iu].mean())

ER = {k: eff_rank(SPACES[k]) for k in names}
PC = {k: pair_cos(SPACES[k]) for k in names}
PC_DEGENERATE = 0.30                 # project convention: see cheat sheet A12
degenerate = {k for k in names if PC[k] > PC_DEGENERATE}
print("\ngeometry per space (pair-cosine is the collapse flag):")
for k in names:
    tag = "  <- DEGENERATE" if k in degenerate else ""
    print(f"  {k:12s} eff.rank {ER[k]:7.1f}/{SPACES[k].shape[1]:5d}   "
          f"pair-cos {PC[k]:+.3f}{tag}")

healthy = [v[2] for kp, v in DELTA.items()
           if not (set(kp) & degenerate)]
rescued = [v[2] for kp, v in DELTA.items() if set(kp) & degenerate]
print(f"\nmean change, ALL pairs          {np.mean(ds):+.3f}  (n={len(ds)})")
print(f"mean change, HEALTHY pairs only {np.mean(healthy):+.3f}  "
      f"(n={len(healthy)}, {sum(1 for d in healthy if d < 0)} negative)")
if rescued:
    print(f"mean change, pairs w/ degenerate {np.mean(rescued):+.3f}  "
          f"(n={len(rescued)})")
if len(ds) > 4:
    from scipy.stats import spearmanr
    kp = list(DELTA)
    rr = spearmanr([max(PC[a], PC[b]) for a, b in kp],
                   [DELTA[k][2] for k in kp]).correlation
    print(f"Spearman(max pair-cosine, improvement) = {rr:+.2f}")
    print("  a strong POSITIVE value means the gain is driven by how")
    print("  degenerate the pair's worst space is, not by hub alignment")

m = float(np.mean(healthy)) if healthy else float(np.mean(ds))
print()
if m > 0.02:
    print("VERDICT: on HEALTHY spaces the hub ACTIVELY ALIGNS local")
    print("structure - transfer works partly because local neighbourhoods")
    print("are brought together.")
elif m < -0.02:
    print("VERDICT: on HEALTHY spaces the hub SPENDS local structure for")
    print("global alignment. The missing ceiling and the width cliff are")
    print("then the same phenomenon.")
else:
    print("VERDICT: on HEALTHY spaces the hub is NEUTRAL on local")
    print("agreement - it relocates structure into shared coordinates")
    print("without creating or destroying it. Transfer therefore runs on")
    print("the GLOBAL linear component, while the local structure G5")
    print("measured rides along intact but unimproved. This explains both")
    print("why the hub works (the global component is real, if thin) and")
    print("why it never reaches the ceiling (the rich local part is not")
    print("what the map transports).")
if rescued and np.mean(rescued) - m > 0.05:
    print()
    print("SEPARATELY: pairs involving a collapsed space gain far more.")
    print("That is the C.11 isotropy rescue reappearing - the hub's")
    print("whitening repairs a degenerate coordinate frame. It is a")
    print("property of the TARGET's geometry, not of the hub aligning")
    print("local structure, and must not be averaged into the headline.")


## 3 · Does the width cliff show up as local-structure loss?

Section C.13 reported that hub width forces a choice: narrow and
components transfer, wide and transfer collapses while reconstruction
improves. That was observed but never explained. If wider hubs preserve
*less* local structure, the cliff has a mechanism.

In [ ]:
import matplotlib.pyplot as plt

WIDTHS = [64, 128, 256, 512, 768]
rows = []
for d in WIDTHS:
    Hd = build_hub(d)
    TH = {k: ridge(SPACES[k], Hd) for k in names}
    IH = {k: l2n(SPACES[k] @ TH[k]) for k in names}
    pres = np.mean([overlap(l2n(SPACES[k][sub]), IH[k][sub]) for k in names])
    cross = np.mean([overlap(IH[a][sub], IH[b][sub])
                     for i, a in enumerate(names) for b in names[i+1:]])
    # ZERO-SHOT transfer, as measured in C.13: fit the head on ONE
    # encoder's hub coordinates, then apply it UNCHANGED to a DIFFERENT
    # encoder's hub coordinates. Fitting and evaluating on the same
    # encoder measures reconstruction, not portability, and does not
    # reproduce the C.13 width cliff - that distinction is the whole
    # point of this cell.
    src = names[0]
    tgt = "txt_bge" if "txt_bge" in names else names[-1]
    unseen = [k for k in names
              if k not in (src, tgt) and k.startswith("img")]
    head = ridge(IH[src], SPACES[tgt])          # fitted on src only
    G = l2n(SPACES[tgt][te])
    def r_at_1(Xh):
        return float(np.mean(np.argmax(l2n(Xh[te] @ head) @ G.T, 1)
                             == np.arange(len(te))))
    r_self = r_at_1(IH[src])                    # reconstruction
    r_zero = (float(np.mean([r_at_1(IH[u]) for u in unseen]))
              if unseen else float("nan"))      # PORTABILITY
    rows.append((d, pres, cross, r_self, r_zero))
    print(f"HUB_DIM {d:4d}   preservation {pres:.3f}   "
          f"cross in-hub {cross:.3f}   self R@1 {r_self:.3f}   "
          f"ZERO-SHOT R@1 {r_zero:.3f}")

W_, P_, C_, R_, Z_ = map(np.array, zip(*rows))
fig, ax = plt.subplots(1, 2, figsize=(11.6, 4.0))
ax[0].plot(W_, P_, "o-", c="#1a5276", label="own neighbourhoods kept")
ax[0].plot(W_, C_, "s-", c="#0f766e", label="cross-encoder overlap")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("HUB_DIM", fontsize=8.4)
ax[0].set_ylabel("kNN overlap (k=10)", fontsize=8.4)
ax[0].set_title("local structure vs hub width", fontsize=10,
                color="#1a1a2e")
ax[0].legend(fontsize=7.4, frameon=False)
ax[1].plot(W_, R_, "o-", c="#94a3b8", label="self (reconstruction)")
ax[1].plot(W_, Z_, "s-", c="#b45309", label="ZERO-SHOT (portability)")
ax[1].legend(fontsize=7.4, frameon=False)
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("HUB_DIM", fontsize=8.4)
ax[1].set_ylabel("transfer R@1", fontsize=8.4)
ax[1].set_title("transfer vs hub width (the C.13 cliff)", fontsize=10,
                color="#1a1a2e")
for x in ax:
    x.grid(alpha=0.2); x.tick_params(labelsize=7.4)
    for s in ("top", "right"): x.spines[s].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.15,
                    wspace=0.26)
plt.show()

from scipy.stats import spearmanr
print(f"\nSpearman(preservation,  ZERO-SHOT transfer) = "
      f"{spearmanr(P_, Z_).correlation:+.2f}")
print(f"Spearman(cross in-hub, ZERO-SHOT transfer) = "
      f"{spearmanr(C_, Z_).correlation:+.2f}")
print("\nNOTE on which curve to read. The C.13 width cliff was measured "
      "on\nZERO-SHOT transfer - a head fitted on one encoder and applied "
      "to\nanother. Self-transfer (fitting and evaluating on the same "
      "encoder)\nmeasures reconstruction instead and does NOT cliff, so "
      "it cannot\ntest the cliff's cause. Only the zero-shot column is "
      "evidence here.")
print(f"\nWith n={len(W_)} widths, |rho| must be near 0.9 to mean "
      "anything.")
print("A strong positive correlation means the width cliff IS a")
print("local-structure loss: wider hubs keep less neighbourhood structure")
print("and transfer less well. That converts an observed trade-off into a")
print("measured mechanism. A weak correlation means the cliff has some")
print("other cause and this explanation should be recorded as FALSIFIED,")
print("in the same way as the four earlier ones.")

## 4 · If not local structure, then what? The spectral account

Section 3 shows the cliff is real and that neither local measure
explains it: both move by their usual small amount across the step where
transfer collapses. A smooth predictor cannot account for a
discontinuous outcome, so the local-structure explanation is falsified
and joins the project's list of tested-and-rejected ones.

Section C.13's original account remains: whitened PCA divides each
direction by its singular value, so components with near-zero variance
are **amplified**. Past some width the hub is dominated by directions
that carry noise rather than shared signal, and cosine geometry — which
weights every direction equally after whitening — is taken over by them.
That mechanism predicts a **threshold** rather than a gradual decline,
which is the shape the data actually has.

This cell measures the amplification directly. The quantity to watch is
1/sqrt(eigenvalue): it is exactly the factor whitening applies to each
direction, so it *is* the noise gain.

In [ ]:
# spectrum of the concatenated, standardised matrix - the object the hub
# is a whitened PCA of
blocks = []
for k in names:
    M = SPACES[k]
    mu, sd = M[tr].mean(0), M[tr].std(0) + 1e-8
    blocks.append((M - mu) / sd)
Zcat = np.hstack(blocks)
Zc = Zcat - Zcat[tr].mean(0)
sv = np.linalg.svd(Zc[tr], compute_uv=False)
evals = sv ** 2 / len(tr)
cum = np.cumsum(evals) / evals.sum()
amp = 1.0 / np.sqrt(np.maximum(evals, 1e-12))

print(f"concatenated width {Zcat.shape[1]}, {len(tr)} train rows\n")
print(f"{'HUB_DIM':>8s} {'sing.val':>10s} {'cum.var':>9s} "
      f"{'amplification':>14s} {'zero-shot R@1':>14s}")
zs_by_w = {d: r for d, _, _, _, r in rows}
for d in WIDTHS:
    if d < len(sv):
        print(f"{d:8d} {sv[d]:10.2f} {cum[d]:9.3f} {amp[d]:14.2f} "
              f"{zs_by_w.get(d, float('nan')):14.3f}")

print("\nThe amplification column is the factor whitening applies to the")
print("LAST direction each hub width admits. Where it rises steeply, the")
print("hub is buying directions whose variance is small enough that")
print("whitening turns them into noise with unit weight.")
print()
print("Read it against the zero-shot column: if the amplification jump")
print("coincides with the transfer collapse, the cliff has a measured")
print("spectral mechanism. If the amplification rises smoothly across")
print("the collapse, this explanation is ALSO unsupported and the cliff")
print("remains unexplained - which is a legitimate thing to report.")

## How to read G7

- **Section 1** says whether the hub map is gentle or violent toward each
  encoder's own local structure. Expect well below 1.0: the map is a
  projection, not a rotation.
- **Section 2** is the decisive one. Three outcomes, three different
  stories about *why the hub works*, all pre-stated above so the verdict
  cannot be chosen after the fact.
- **Section 3** tests whether the C.13 width cliff has a local-structure
  mechanism. If the correlation is weak, that explanation joins the
  project's list of falsified ones rather than being quietly dropped.

**What G7 cannot show.** It measures what the hub does to structure, not
whether a better hub exists. A non-linear or neighbourhood-preserving
embedding might carry local structure where whitened PCA does not; this
notebook does not test that, and the project's "no training" constraint
rules it out by design. That is a named limitation, not an oversight.